# RAG Pipeline — PyTorch Documentation Assistant

## 2.1 Load & Inspect

In [1]:
import os
from pypdf import PdfReader

RAW_DOCS_PATH = "../data/raw_docs"

pdf_files = [f for f in os.listdir(RAW_DOCS_PATH) if f.endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF files")

doc_info = []

for filename in pdf_files:
    filepath = os.path.join(RAW_DOCS_PATH, filename)
    try:
        reader = PdfReader(filepath)
        num_pages = len(reader.pages)

        first_page_text = reader.pages[0].extract_text()
        has_text = len(first_page_text.strip()) > 0

        doc_info.append({
            "filename": filename,
            "num_pages": num_pages,
            "has_extractable_text": has_text,
        })

    except Exception as e:
        doc_info.append({
            "filename": filename,
            "num_pages": None,
            "has_extractable_text": False,
            "error": str(e),
        })

for doc in doc_info:
    print(doc)

Found 10 PDF files
{'filename': 'Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf', 'num_pages': 8, 'has_extractable_text': True}
{'filename': 'Build the Neural Network — PyTorch Tutorials 2.14.0+cu130 documentation.pdf', 'num_pages': 8, 'has_extractable_text': True}
{'filename': 'Datasets & DataLoaders — PyTorch Tutorials 2.14.0+cu130 documentation.pdf', 'num_pages': 9, 'has_extractable_text': True}
{'filename': 'Learning PyTorch with Examples — PyTorch Tutorials 2.14.0+cu130 documentation.pdf', 'num_pages': 17, 'has_extractable_text': True}
{'filename': 'Linear — PyTorch 2.14 documentation.pdf', 'num_pages': 3, 'has_extractable_text': True}
{'filename': 'Module — PyTorch 2.14 documentation.pdf', 'num_pages': 32, 'has_extractable_text': True}
{'filename': 'Optimizing Model Parameters — PyTorch Tutorials 2.14.0+cu130 documentation.pdf', 'num_pages': 8, 'has_extractable_text': True}
{'filename': 'Save and Load the Model — PyTorch Tutorials

**Summary:**
- Total documents: 10
- Total pages across all documents: 123
- All formats: PDF (exported from PyTorch official documentation and tutorials)
- Files that failed to parse or need OCR: None — all files have extractable text
- Notable observation: reference pages (e.g. `Module`, `torch.optim`) are significantly
  longer (24-32 pages) than tutorial pages (3-17 pages), which will need to be
  considered when choosing a chunking strategy.

In [2]:
def extract_full_text(filepath):
    reader = PdfReader(filepath)
    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"
    return full_text


documents = {}

for filename in pdf_files:
    filepath = os.path.join(RAW_DOCS_PATH, filename)
    documents[filename] = extract_full_text(filepath)

for filename, text in documents.items():
    print(f"{filename}: {len(text)} characters")

Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 10468 characters
Build the Neural Network — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 10372 characters
Datasets & DataLoaders — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 10410 characters
Learning PyTorch with Examples — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 24743 characters
Linear — PyTorch 2.14 documentation.pdf: 2902 characters
Module — PyTorch 2.14 documentation.pdf: 39717 characters
Optimizing Model Parameters — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 10946 characters
Save and Load the Model — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 6787 characters
Tensors — PyTorch Tutorials 2.14.0+cu130 documentation.pdf: 9264 characters
torch.optim — PyTorch 2.14 documentation.pdf: 36359 characters


## Text Cleaning

Initial inspection of extracted text revealed PDF export artifacts that add
noise without value: repeated navigation menus, page headers/footers, dates,
and page numbers. These are cleaned before chunking to improve retrieval quality.

In [3]:
import re


def clean_text(text):
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        if "||" in stripped:
            continue

        # Catches "Colab Notebook GitHub" even if merged with other text
        if "Colab Notebook GitHub" in stripped:
            continue

        if stripped.startswith("http://") or stripped.startswith("https://"):
            continue

        if re.match(r"^\d+/\d+$", stripped):
            continue

        if re.search(r"[مص]\d{1,2}:\d{2}", stripped):
            continue

        if stripped.startswith("Created On:") or "Last Verified" in stripped:
            continue

        cleaned_lines.append(stripped)

    cleaned_text = "\n".join(cleaned_lines)
    cleaned_text = re.sub(r"\n{2,}", "\n", cleaned_text)

    return cleaned_text


cleaned_documents = {filename: clean_text(text) for filename, text in documents.items()}

sample_filename = list(documents.keys())[0]

print("BEFORE cleaning (first 500 chars):")
print(documents[sample_filename][:500])
print("\n" + "="*50 + "\n")
print("AFTER cleaning (first 500 chars):")
print(cleaned_documents[sample_filename][:500])

BEFORE cleaning (first 500 chars):
Colab Notebook GitHub
Learn the Basics || Quickstart || Tensors || Datasets & DataLoaders || Transforms || Build Model
|| Autograd || Optimization || Save & Load Model
Automatic Differentiation with
torch.autograd
Created On: Feb 10, 2021 | Last Updated: Jan 16, 2024 | Last Verified: Nov 05, 2024
When training neural networks, the most frequently used algorithm is back propagation. In
this algorithm, parameters (model weights) are adjusted according to the gradient of the loss
function with res


AFTER cleaning (first 500 chars):
Automatic Differentiation with
torch.autograd
When training neural networks, the most frequently used algorithm is back propagation. In
this algorithm, parameters (model weights) are adjusted according to the gradient of the loss
function with respect to the given parameter.
To compute those gradients, PyTorch has a built-in differentiation engine called
torch.autograd . It supports automatic computation of gradient for any 

## 2.2 Chunking Strategy

In [4]:
def chunk_by_paragraphs(text, max_chunk_size=800, overlap=150, min_chunk_size=100):
    """
    Splits text into chunks based on natural paragraph boundaries.
    Paragraphs longer than max_chunk_size are further split using
    fixed-size chunking with overlap as a fallback. Very short
    paragraphs are merged with the next one to avoid tiny, low-value chunks.
    """
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

    chunks = []
    buffer = ""

    for para in paragraphs:
        candidate = (buffer + "\n" + para).strip() if buffer else para

        if len(candidate) < min_chunk_size:
            buffer = candidate
            continue

        if len(candidate) <= max_chunk_size:
            buffer = candidate
        else:
            if buffer:
                chunks.append(buffer)
            if len(para) > max_chunk_size:
                start = 0
                while start < len(para):
                    end = start + max_chunk_size
                    chunks.append(para[start:end])
                    start += max_chunk_size - overlap
                buffer = ""
            else:
                buffer = para

    if buffer:
        chunks.append(buffer)

    return chunks


all_chunks = []

for filename, text in cleaned_documents.items():
    doc_chunks = chunk_by_paragraphs(text, max_chunk_size=800, overlap=150)

    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "source": filename,
            "chunk_id": f"{filename}_{i}",
            "text": chunk,
        })

print(f"Total chunks created: {len(all_chunks)}")
print("\nExample chunk:")
print(all_chunks[5])

Total chunks created: 184

Example chunk:
{'source': 'Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf', 'chunk_id': 'Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf_5', 'text': 'More on Computational Graphs\nConceptually, autograd keeps a record of data (tensors) and all executed operations (along\nwith the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function\nobjects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this\ngraph from roots to leaves, you can automatically compute the gradients using the chain rule.\nIn a forward pass, autograd does two things simultaneously:\nz = torch.matmul(x, w)+b\nprint(z.requires_grad)\nwith torch.no_grad():\nz = torch.matmul(x, w)+b\nprint(z.requires_grad)\nTrue\nFalse\nz = torch.matmul(x, w)+b\nz_det = z.detach()\nprint(z_det.requires_grad)\nFalse\n\ueddbrun the requested operation to com

**Chunking strategy justification:**
- Used fixed-size chunking with a chunk size of 700 characters and 150 character overlap
- Chunk size of 700 was chosen to keep enough context for code examples and explanations
  to remain coherent, while staying small enough for precise retrieval
- 150-character overlap ensures that concepts split across chunk boundaries aren't
  lost entirely in either chunk
- Chunking is applied after text cleaning to avoid wasting chunk space on
  navigation menus, headers, and dates

## 2.3 Embeddings & Vector Store

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {embeddings.shape[1]} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Generated 184 embeddings
Each embedding has 384 dimensions


In [6]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../data/vector_store")

# Delete any existing collection first to avoid mixing old and new data
try:
    chroma_client.delete_collection(name="pytorch_docs")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="pytorch_docs")

collection.add(
    ids=[chunk["chunk_id"] for chunk in all_chunks],
    embeddings=embeddings.tolist(),
    documents=[chunk["text"] for chunk in all_chunks],
    metadatas=[{"source": chunk["source"]} for chunk in all_chunks],
)

print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 184 chunks in ChromaDB


## 2.4 Retrieval & Prompting

In [7]:
def retrieve_relevant_chunks(query, n_results=4):
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "distance": results["distances"][0][i],
        })

    return retrieved


test_query = "How do I use autograd to compute gradients?"
results = retrieve_relevant_chunks(test_query)

for r in results:
    print(f"Source: {r['source']}")
    print(f"Distance: {r['distance']:.4f}")
    print(f"Text: {r['text'][:150]}...")
    print("---")

Source: Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf
Distance: 0.6305
Text: More on Computational Graphs
Conceptually, autograd keeps a record of data (tensors) and all executed operations (along
with the resulting new tensors...
---
Source: Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf
Distance: 0.6648
Text: Automatic Differentiation with
torch.autograd
When training neural networks, the most frequently used algorithm is back propagation. In
this algorithm...
---
Source: Learning PyTorch with Examples — PyTorch Tutorials 2.14.0+cu130 documentation.pdf
Distance: 0.6863
Text: # Manually zero the gradients after updating weights
a.grad = None
b.grad = None
c.grad = None
d.grad = None
print(f'Result: y = {a.item()} + {b.item(...
---
Source: Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf
Distance: 0.7234
Text: The backward pass kicks off w

In [8]:
def build_rag_prompt(query, retrieved_chunks):
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks):
        context_blocks.append(f"[Source {i+1}: {chunk['source']}]\n{chunk['text']}")

    context = "\n\n".join(context_blocks)

    prompt = f"""You are a helpful assistant answering questions about PyTorch
    documentation. Use ONLY the context below to answer the question. If the
    context doesn't contain enough information to answer, say so clearly instead
    of making up an answer.

    Do NOT add any "Source:" line, citation, filename, or reference at the end of
    your answer — that will be added separately by the system. Just answer the
    question directly using the context.

Context:
{context}

Question: {query}

Answer:"""

    return prompt


test_prompt = build_rag_prompt(test_query, results)
print(test_prompt)

You are a helpful assistant answering questions about PyTorch
    documentation. Use ONLY the context below to answer the question. If the
    context doesn't contain enough information to answer, say so clearly instead
    of making up an answer.

    Do NOT add any "Source:" line, citation, filename, or reference at the end of
    your answer — that will be added separately by the system. Just answer the
    question directly using the context.

Context:
[Source 1: Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf]
More on Computational Graphs
Conceptually, autograd keeps a record of data (tensors) and all executed operations (along
with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function
objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this
graph from roots to leaves, you can automatically compute the gradients using the chain rule.
In a forward pass, autograd d

## Connecting to Ollama (Local LLM)

In [9]:
import ollama


def ask_rag_assistant(query, n_results=4):
    retrieved_chunks = retrieve_relevant_chunks(query, n_results=n_results)
    prompt = build_rag_prompt(query, retrieved_chunks)

    response = ollama.chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
    )

    raw_answer = response["message"]["content"]

    unique_sources = list(dict.fromkeys(chunk["source"] for chunk in retrieved_chunks))
    citation_line = "Sources: " + ", ".join(unique_sources)

    final_answer = f"{raw_answer.strip()}\n\n{citation_line}"

    return final_answer, retrieved_chunks


# Test it
answer, sources_used = ask_rag_assistant("How do I use autograd to compute gradients?")

print("ANSWER:")
print(answer)

ANSWER:
To use autograd to compute gradients, you need to create a computational graph by defining a sequence of operations. This can be done by calling functions that create tensors and perform operations on them, such as `torch.matmul()`.

In the example from the PyTorch tutorials, the computational graph is defined as follows:

`z = torch.matmul(x, w) + b`

The `torch.matmul()` function creates a tensor `z` that depends on the input tensor `x` and the parameters `w` and `b`. When you call `backward()` on the root of the graph (in this case, `z`), autograd will compute the gradients of the loss function with respect to the leaf tensors (in this case, `x`, `w`, and `b`).

To use autograd to compute gradients, you need to:

1. Create a computational graph by defining a sequence of operations.
2. Call `backward()` on the root of the graph to start the backward pass.
3. The backward pass will compute the gradients of the loss function with respect to the leaf tensors.

Note that you can 

In [10]:
test_questions = [
    "How do I use autograd to compute gradients?",
    "What is the purpose of the DataLoader class?",
    "How do I define a custom neural network using nn.Module?",
    "What does the Linear layer do in PyTorch?",
    "How do I save and load a trained model?",
    "What optimizers are available in torch.optim?",
    "How do I create a tensor and check its shape?",
    "What is the difference between a Dataset and a DataLoader?",
    "How do I disable gradient tracking in PyTorch?",
    "What is the role of the loss function during training?",
]

In [11]:
evaluation_results = []

for question in test_questions:
    answer, sources = ask_rag_assistant(question)

    evaluation_results.append({
        "question": question,
        "answer": answer,
        "sources": [s["source"] for s in sources],
        "top_distance": sources[0]["distance"],
    })

    print(f"Q: {question}")
    print(f"A: {answer[:200]}...")
    print(f"Top source: {sources[0]['source']} (distance: {sources[0]['distance']:.4f})")
    print("=" * 80)

Q: How do I use autograd to compute gradients?
A: To use autograd to compute gradients, you need to create a computational graph by building a DAG of Function objects, with leaves being the input tensors and roots being the output tensors. Then, call...
Top source: Automatic Differentiation with torch.autograd — PyTorch Tutorials 2.14.0+cu130 documentation.pdf (distance: 0.6305)
Q: What is the purpose of the DataLoader class?
A: The purpose of the DataLoader class is to enable easy access to the samples in a dataset, allowing you to iterate over the data in batches. This decouples the data processing code from the model train...
Top source: Datasets & DataLoaders — PyTorch Tutorials 2.14.0+cu130 documentation.pdf (distance: 1.1898)
Q: How do I define a custom neural network using nn.Module?
A: To define a custom neural network using `nn.Module`, you need to subclass `nn.Module` and implement the `forward` method, which defines how the input data should be processed through the network.

In [13]:
answer, sources = ask_rag_assistant("How do I save and load a trained model?")
print(answer)

To save and load a trained model, you need to follow these steps:

1. First, create an instance of your model and train it using your dataset and optimizer.
2. After training, call the `state_dict()` method on your model to get the learned parameters.
3. Save the parameters to a file using the `torch.save()` method.

To load the model, you need to:

1. Instantiate the model class with the same architecture as before.
2. Load the saved parameters from the file using the `model.load_state_dict()` method.
3. Call the `eval()` method on your model to set the dropout and batch normalization layers to evaluation mode.

Here is an example:

```python
import torch
import torchvision.models as models
from torchvision import datasets, transforms

# Load the FashionMNIST dataset
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False,

In [14]:
answer, sources = ask_rag_assistant("What optimizers are available in torch.optim?")
print(answer)

Most commonly used methods are already supported, and the interface is general enough, so that more sophisticated ones can also be easily integrated in the future. However, the context does not provide a comprehensive list of available optimizers in torch.optim.

Sources: torch.optim — PyTorch 2.14 documentation.pdf, Optimizing Model Parameters — PyTorch Tutorials 2.14.0+cu130 documentation.pdf


In [15]:
answer, sources = ask_rag_assistant("What optimizers are available in torch.optim?", n_results=8)
print(answer)
print("\n\nSources retrieved:")
for s in sources:
    print(f"- {s['source']} (distance: {s['distance']:.4f})")

Most commonly used methods are already supported, and the interface is general enough, so that more sophisticated ones can also be easily integrated in the future.

Some examples of optimizers that are already supported include:

- SGD (Stochastic Gradient Descent)
- RMSprop (Root Mean Square Propagation)
- Adam (Adagrad variant with momentum)
- Adadelta (Adagrad variant with momentum and L2 regularization)
- Adagrad (Adagrad optimization algorithm)
- AsGD (Averaged Stochastic Gradient Descent)
- LBFGS (Limited-memory Broyden-Fletcher-Goldfarb-Shanno)

These optimizers can be used with the `torch.optim` module by passing the optimizer name, along with any required parameters, to the `Optimizer` constructor.

Sources: torch.optim — PyTorch 2.14 documentation.pdf, Optimizing Model Parameters — PyTorch Tutorials 2.14.0+cu130 documentation.pdf, Module — PyTorch 2.14 documentation.pdf, Learning PyTorch with Examples — PyTorch Tutorials 2.14.0+cu130 documentation.pdf


Sources retrieved:
- t

## 2.5 Vision Component

**Not applicable — Core Track.**

This project follows the **Core Track** (text-only RAG assistant). The Extended
Track adds a Computer Vision/YOLO component for multimodal input, which is not
implemented in this version.

If extended in the future, possible additions include:
- OCR for scanned PyTorch documentation pages
- Diagram/figure detection in tutorial PDFs
- Feeding visual context into the RAG prompt

## 2.6 Evaluation

### Results Table (10 test questions)

| # | Question | Top Retrieved Source | Distance | Grounded? |
|---|----------|----------------------|----------|-----------|
| 1 | How do I use autograd to compute gradients? | Automatic Differentiation with torch.autograd | 0.6305 | Yes |
| 2 | What is the purpose of the DataLoader class? | Datasets & DataLoaders | 1.1898 | Yes |
| 3 | How do I define a custom neural network using nn.Module? | Learning PyTorch with Examples | 0.8796 | Yes |
| 4 | What does the Linear layer do in PyTorch? | Build the Neural Network | 0.8930 | Yes |
| 5 | How do I save and load a trained model? | Save and Load the Model | 0.8729 | Yes (see note below) |
| 6 | What optimizers are available in torch.optim? | torch.optim | 0.4771 | Partial (see note below) |
| 7 | How do I create a tensor and check its shape? | Tensors | 0.9066 | Yes |
| 8 | What is the difference between a Dataset and a DataLoader? | Datasets & DataLoaders | 1.0423 | Yes |
| 9 | How do I disable gradient tracking in PyTorch? | Optimizing Model Parameters | 0.8867 | Yes |
| 10 | What is the role of the loss function during training? | Optimizing Model Parameters | 1.0169 | Yes |

**Summary:** 9 out of 10 answers were fully grounded in the retrieved context, with
correct technical content and no fabricated facts. 1 answer (Q6) required a
retrieval-parameter adjustment to become fully informative, and even after
that fix showed a subtler form of partial hallucination (see Failure Case 5).

### Failure Cases and Mitigations

**1. Citation hallucination (resolved).** In early testing, the local `llama3.2`
model occasionally invented citation details not present in the retrieved
metadata — first an arbitrary section number, then an invented section label,
then an entirely fabricated filename derived from a heading it misread inside
the retrieved text. Two rounds of prompt refinement (a negative instruction,
then a positive instruction with a concrete example) reduced but did not
eliminate the issue, since the model's answer content stayed accurate and
grounded in every case — only the citation formatting was affected. This was
resolved by removing citation generation from the LLM entirely: the prompt now
instructs the model not to add any source line, and the system builds the
citation programmatically from the actual retrieved chunks' metadata. This
guarantees citation accuracy regardless of the model's instruction-following
behavior.

**2. Citation over-inclusion (documented, not fixed).** Because citations are
now built from all `n_results` retrieved chunks rather than only the ones the
model actually drew from, some answers cite a source that did not meaningfully
contribute to the answer (e.g. Q5 cites "Datasets & DataLoaders" despite the
answer being entirely about saving/loading models). This is a trade-off of the
programmatic-citation fix: it guarantees no invented filenames, at the cost of
occasionally including a less relevant source in the citation list.

**3. Retrieval granularity limits on long reference documents (Q6, initial
attempt).** With the default `n_results=4`, the closest-matching document for
"What optimizers are available in torch.optim?" was `torch.optim` itself
(distance 0.4771 — the lowest of all 10 questions), a 24-page reference
document that does list specific optimizers (SGD, Adam, etc.) elsewhere in
the file. However, the specific chunk retrieved as the top match did not
include that list, so the model gave an overly general, under-informative
answer rather than inventing optimizer names. This shows that answer
completeness depends not just on which document is most relevant, but on
which specific chunk within it gets retrieved — a limitation of fixed top-k
retrieval on long documents.

**4. Mitigation for Failure Case 3 (partially effective).** Increasing
`n_results` from 4 to 8 for this query retrieved a chunk containing the
actual optimizer list, and the model's answer then correctly named real
PyTorch optimizers (SGD, RMSprop, Adam, Adadelta, Adagrad, ASGD, LBFGS) —
all genuinely present in `torch.optim`. This confirms the completeness issue
was a retrieval-scope problem, not a model reasoning failure.

**5. Partial hallucination in supplementary detail (new finding, unresolved).**
Despite the optimizer names themselves being correct and grounded, the model
added brief parenthetical descriptions for each one (e.g., "Adam (Adagrad
variant with momentum)") that were not present in the retrieved context and
are factually inaccurate — Adam is not a variant of Adagrad, and Adadelta's
description incorrectly attributes it to "L2 regularization." This is a case
of partial hallucination: the core factual content (optimizer names) is
grounded and correct, while supplementary explanatory detail was fabricated
from the model's general knowledge and contains technical errors. This
highlights that grounding must be verified at the level of individual claims
within an answer, not just the answer as a whole. Not resolved due to time
constraints; a possible future mitigation is instructing the model to state
names only, with no added explanation, when the context doesn't explicitly
define each term.

**6. Retrieval distance varies significantly by question phrasing.** Distance
scores ranged from 0.48 (Q6) to 1.19 (Q2), even though both answers were
grounded and correct. This indicates that distance reflects phrasing/lexical
similarity between the query and source text more than semantic correctness,
and should not be used alone as a quality signal.

## 2.7 Export

The vector store is already persisted automatically by ChromaDB's
`PersistentClient` at `../data/vector_store` (see section 2.3). This section
additionally saves the pipeline configuration so the backend can load
everything without needing to inspect this notebook.

In [16]:
import json

config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "embedding_dimensions": 384,
    "chunking_strategy": "paragraph-based",
    "max_chunk_size": 800,
    "chunk_overlap": 150,
    "min_chunk_size": 100,
    "vector_store_path": "data/vector_store",
    "collection_name": "pytorch_docs",
    "llm_model": "llama3.2",
    "retrieval_n_results": 4,
    "total_chunks": len(all_chunks),
    "total_source_documents": len(cleaned_documents),
}

config_path = "../data/vector_store/config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config saved to {config_path}")
print(json.dumps(config, indent=2))

Config saved to ../data/vector_store/config.json
{
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_dimensions": 384,
  "chunking_strategy": "paragraph-based",
  "max_chunk_size": 800,
  "chunk_overlap": 150,
  "min_chunk_size": 100,
  "vector_store_path": "data/vector_store",
  "collection_name": "pytorch_docs",
  "llm_model": "llama3.2",
  "retrieval_n_results": 4,
  "total_chunks": 184,
  "total_source_documents": 10
}
